In [1]:
# ==============================================================================
# PROJETO: Inteligência em Planejamento de Demanda & S&OP (End-to-End Analytics)
# ETAPA: Pipeline de Ingestão, Tratamento e Análise Prescritiva de Demanda
# ENGINE: PySpark / Spark SQL
# ==============================================================================

# 1. INSTALAÇÃO E INICIALIZAÇÃO DA SESSÃO SPARK
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, abs, when, sum, avg

# Criação da SparkSession
spark = SparkSession.builder \
    .appName("SOP_Demand_Planning_Analytics") \
    .getOrCreate()

print("--> Sessão PySpark iniciada com sucesso.")

# ------------------------------------------------------------------------------
# 2. CARREGAMENTO E MOCK DA BASE DE DADOS (SIMULAÇÃO DE EXTRAÇÃO DO ERP/DATA LAKE)
# ------------------------------------------------------------------------------
data = [
    ("SKU-PERF-001", "Perfumaria", 10000, 6200, 7000, 6800),
    ("SKU-PERF-002", "Perfumaria", 5000, 5800, 6000, 5800),
    ("SKU-MAKE-101", "Maquiagem", 15000, 14200, 15000, 14200),
    ("SKU-CORP-201", "Corpo&Banho", 8000, 3100, 4000, 3800),
    ("SKU-CABO-301", "Cabelos", 12000, 11800, 12000, 11800)
]

columns = ["SKU", "Categoria", "Forecast_Un", "Sell_Out_Un", "Sell_In_Un", "Atendido_Un"]

# Criando o DataFrame Spark
df_raw = spark.createDataFrame(data, columns)

# ------------------------------------------------------------------------------
# 3. ENGENHARIA DE MÉTRICAS E INDICADORES DE ACCURACY & SUPPLY CHAIN
# ------------------------------------------------------------------------------
df_metrics = df_raw \
    .withColumn("Erro_Absoluto", abs(col("Sell_Out_Un") - col("Forecast_Un"))) \
    .withColumn("BIAS_Unidades", col("Forecast_Un") - col("Sell_Out_Un")) \
    .withColumn("Erro_Pct_SKU", round((col("Erro_Absoluto") / col("Sell_Out_Un")) * 100, 1)) \
    .withColumn("Fill_Rate_Pct", round((col("Atendido_Un") / col("Sell_In_Un")) * 100, 2))

# ------------------------------------------------------------------------------
# 4. REGRAS PRESCRITIVAS (MOTOR DE DECISÃO S&OP / S&OE)
# ------------------------------------------------------------------------------
# Direciona SKUs com sobredemanda/subdemanda para os times de negócio responsáveis
df_prescritivo = df_metrics.withColumn(
    "Acao_Prescritiva",
    when(col("BIAS_Unidades") > 2000, "Trade Mkt & RGM")      # Viés de sobre-estoque crítico
    .when(col("BIAS_Unidades") < -500, "Supply Chain & Fábrica") # Viés de ruptura / falta
    .otherwise("Planejamento Continuo")                       # Dentro do nível tolerável
)

# ------------------------------------------------------------------------------
# 5. CONSOLIDAÇÃO E EXPOSIÇÃO DOS RESULTADOS PROCESSADOS
# ------------------------------------------------------------------------------
print("\n=== VISÃO DETALHADA POR SKU (ENGENHARIA DE MÉTICAS S&OP) ===")
df_prescritivo.select(
    "SKU", "Categoria", "Forecast_Un", "Sell_Out_Un",
    "BIAS_Unidades", "Erro_Pct_SKU", "Fill_Rate_Pct", "Acao_Prescritiva"
).show()

# Exportação simulada do resultado consolidado (para consumo no Looker Studio)
# df_prescritivo.write.csv("saida_sop_dados_tratados.csv", header=True, mode="overwrite")

--> Sessão PySpark iniciada com sucesso.

=== VISÃO DETALHADA POR SKU (ENGENHARIA DE MÉTICAS S&OP) ===
+------------+-----------+-----------+-----------+-------------+------------+-------------+--------------------+
|         SKU|  Categoria|Forecast_Un|Sell_Out_Un|BIAS_Unidades|Erro_Pct_SKU|Fill_Rate_Pct|    Acao_Prescritiva|
+------------+-----------+-----------+-----------+-------------+------------+-------------+--------------------+
|SKU-PERF-001| Perfumaria|      10000|       6200|         3800|        61.3|        97.14|     Trade Mkt & RGM|
|SKU-PERF-002| Perfumaria|       5000|       5800|         -800|        13.8|        96.67|Supply Chain & Fá...|
|SKU-MAKE-101|  Maquiagem|      15000|      14200|          800|         5.6|        94.67|Planejamento Cont...|
|SKU-CORP-201|Corpo&Banho|       8000|       3100|         4900|       158.1|         95.0|     Trade Mkt & RGM|
|SKU-CABO-301|    Cabelos|      12000|      11800|          200|         1.7|        98.33|Planejamento Co